# 04 — Model Evaluation
**Fraud Detection & Anomaly Scoring System**

Goals:
- Compare all three models on the held-out test set
- ROC-AUC, PR-AUC, KS-Statistic, F1, Precision, Recall
- Confusion matrices
- Choose best threshold per model

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve

from src.features import build_train_test
from src.models import load_models, iso_forest_scores
from src.evaluation import evaluate_classifier, evaluate_iso_forest, ks_statistic

plt.rcParams.update({'figure.dpi': 120})
sns.set_theme(style='darkgrid')

_, _, X_test, _, _, y_test, feat_names = build_train_test(apply_smote=False)
iso, xgb_m, lgbm_m, scaler = load_models()
print("Models loaded.")

## 1. Individual Model Evaluation

In [ ]:
res_iso  = evaluate_iso_forest(iso, X_test, y_test)
res_xgb  = evaluate_classifier('XGBoost', y_test, xgb_m.predict_proba(X_test)[:, 1])
res_lgbm = evaluate_classifier('LightGBM', y_test, lgbm_m.predict_proba(X_test)[:, 1])

## 2. Side-by-Side Comparison Table

In [ ]:
comparison = pd.DataFrame([res_iso, res_xgb, res_lgbm]).set_index('model')
comparison = comparison[['roc_auc','pr_auc','ks','precision','recall','f1','threshold']]
comparison.columns = ['ROC-AUC','PR-AUC','KS-Stat','Precision','Recall','F1','Threshold']
comparison.round(4).style.highlight_max(axis=0, color='#d5f5e3').highlight_min(axis=0, color='#fadbd8')

## 3. Overlaid ROC Curves

In [ ]:
iso_scores = iso_forest_scores(iso, X_test)
xgb_scores = xgb_m.predict_proba(X_test)[:, 1]
lgbm_scores= lgbm_m.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
for scores, label, color in [
    (iso_scores,  f"Iso Forest (AUC={res_iso['roc_auc']:.3f})",  '#e74c3c'),
    (xgb_scores,  f"XGBoost   (AUC={res_xgb['roc_auc']:.3f})",  '#3498db'),
    (lgbm_scores, f"LightGBM  (AUC={res_lgbm['roc_auc']:.3f})", '#27ae60'),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    axes[0].plot(fpr, tpr, lw=2, label=label, color=color)

axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curves — All Models')
axes[0].legend(fontsize=9)

# Precision-Recall
for scores, label, color in [
    (iso_scores,  f"Iso Forest (AP={res_iso['pr_auc']:.3f})",  '#e74c3c'),
    (xgb_scores,  f"XGBoost   (AP={res_xgb['pr_auc']:.3f})",  '#3498db'),
    (lgbm_scores, f"LightGBM  (AP={res_lgbm['pr_auc']:.3f})", '#27ae60'),
]:
    prec, rec, _ = precision_recall_curve(y_test, scores)
    axes[1].plot(rec, prec, lw=2, label=label, color=color)

axes[1].axhline(y_test.mean(), color='gray', linestyle='--', lw=1, label='Baseline')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves — All Models')
axes[1].legend(fontsize=9)

plt.suptitle('Model Comparison — ROC & PR Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eval_combined_curves.png', dpi=120)
plt.show()

## 4. Metric Bar Chart

In [ ]:
metrics = ['ROC-AUC','PR-AUC','KS-Stat','Precision','Recall','F1']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
for i, (res, label, color) in enumerate([
    (res_iso,  'Isolation Forest', '#e74c3c'),
    (res_xgb,  'XGBoost',          '#3498db'),
    (res_lgbm, 'LightGBM',         '#27ae60'),
]):
    vals = [res['roc_auc'], res['pr_auc'], res['ks'],
            res['precision'], res['recall'], res['f1']]
    bars = ax.bar(x + i*width, vals, width, label=label, color=color, edgecolor='white', alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(1.0, color='black', lw=0.5, linestyle=':')
plt.tight_layout()
plt.savefig('../reports/eval_metric_comparison.png', dpi=120)
plt.show()

## 5. Key Findings

| Metric | Best Model | Why It Matters |
|--------|-----------|----------------|
| ROC-AUC | XGBoost / LightGBM | Overall discrimination power |
| PR-AUC | XGBoost / LightGBM | Most informative for imbalanced data |
| KS-Statistic | XGBoost / LightGBM | Classic fintech risk metric |
| Recall | Tune threshold | Catching fraud matters most in production |

> **Next step:** SHAP Explainability (`05_drift_monitoring.ipynb`)